# 10 — Prensa online MediaCloud: Inmigración

**Objetivo:** construir un dataset limpio de noticias sobre inmigración y extraer el cuerpo completo de cada noticia a partir de las URLs de MediaCloud.

En este notebook **no hacemos NLP todavía**. Primero cerramos la capa de datos:

1. Cargar CSV de MediaCloud.
2. Limpiar columnas y fechas.
3. Revisar duplicados.
4. Probar extracción de texto con Trafilatura.
5. Medir tasa de éxito.
6. Extraer cuerpos de noticias.
7. Guardar dataset procesado.


## 1. Instalación e importación de librerías

In [1]:
import pandas as pd
import numpy as np
import trafilatura
from tqdm import tqdm
tqdm.pandas()
from pathlib import Path
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

In [ ]:
from pathlib import Path

BASE = Path(
    r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13"
)

for archivo in BASE.rglob("2015-lgtb.csv"):
    print(archivo)

C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\PRENSA\RAW\2015-lgtb.csv


## 2. Definir rutas

Cambia estas rutas según dónde guardes el CSV descargado de MediaCloud.

In [ ]:
from pathlib import Path

# ============================================================
# PARÁMETROS
# ============================================================

YEAR = 2023
TEMA = "lgtb"

# ============================================================
# RUTAS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13"
)

RUTA_RAW_DIR = (
    PROJECT_ROOT
    / "01_data"
    / "01_raw"
    / "prensa"
)

RUTA_PROCESADOS = (
    PROJECT_ROOT
    / "01_data"
    / "02_processed"
    / "prensa"
    / "articulos_con_texto"
)

# Los archivos de inmigración incluyen el sufijo query2.
if TEMA == "inmigracion":
    nombre_entrada = f"{YEAR}-inmigracion-query2.csv"
else:
    nombre_entrada = f"{YEAR}-lgtb.csv"

RUTA_RAW = RUTA_RAW_DIR / nombre_entrada

RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)

RUTA_SALIDA = (
    RUTA_PROCESADOS
    / f"{TEMA}_{YEAR}_con_texto.csv"
)

print("Entrada:", RUTA_RAW)
print("Existe:", RUTA_RAW.exists())
print("Salida:", RUTA_SALIDA)

C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\PRENSA\RAW\2023-lgtb.csv
True


## 3. Cargar CSV de MediaCloud

In [4]:
df = pd.read_csv(RUTA_RAW)

print(df.shape)
print(df.columns.tolist())

(79141, 8)
['id', 'indexed_date', 'language', 'media_name', 'media_url', 'publish_date', 'title', 'url']


## 4. Revisión inicial de columnas

In [5]:
df.columns

Index(['id', 'indexed_date', 'language', 'media_name', 'media_url',
       'publish_date', 'title', 'url'],
      dtype='object')

## 5. Limpieza básica

Nos quedamos con las columnas útiles: fecha, medio, título y URL.

In [6]:
columnas_utiles = ['id', 'publish_date', 'media_name', 'media_url', 'title', 'url', 'language']

columnas_existentes = [col for col in columnas_utiles if col in df.columns]
df = df[columnas_existentes].copy()

# Convertir fecha
df['publish_date'] = pd.to_datetime(df['publish_date'], errors='coerce')

# Crear año y mes
df['anio'] = df['publish_date'].dt.year
df['mes'] = df['publish_date'].dt.month

# Limpiar textos básicos
df['title'] = df['title'].astype(str).str.strip()
df['url'] = df['url'].astype(str).str.strip()
df['media_name'] = df['media_name'].astype(str).str.strip()

df.head()

,id,publish_date,media_name,media_url,title,url,language,anio,mes
0,bf59f4d007847e74242a26f2f854e3d1ca34080261092b6ea5ef8a82df2204fd,2023-07-03,rtve.es,rtve.es,Daniela Vega es una mujer fantástica': curiosidades de la película ganadora del Premio Oscar,https://www.rtve.es/play/noticias/20230703/mujer-fantastica-pelicula-curiosidades-lgtbi/2450142.shtml,es,2023,7
1,8db3b4bbe8cb969f6b31aef9156732982f6150a23cd63c83798a2c89c365d49b,2023-11-03,elperiodico.cat,elperiodico.cat,Detingut a Terrassa (Barcelona) per difondre postulats gihadistes i amenaçar el col·lectiu jueu i LGTBI,https://www.elperiodico.cat/politica/20231103/detingut-gihadista-terrassa-94172840?utm_source=rss-noticias&utm_mediu...,ca,2023,11
2,8ded3f9573aec177adc2b6c52e492849c179d860d1d2ab5e8db9328f4de77772,2023-06-28,melillahoy.es,melillahoy.es,El PSOE ve “muy mal” la ausencia de CPM en el Día del Orgullo LGTBI,https://melillahoy.es/el-psoe-ve-muy-mal-la-ausencia-de-cpm-en-el-dia-del-orgullo-lgtbi/#comment-5355,es,2023,6
3,1c1cdcba71aeb2f5d3c91df0b25129a8b2c6859425c1cf31e40a0904ea395433,2023-12-19,ondacero.es,ondacero.es,El Gabinete: ¿qué implica que la Iglesia acepte bendecir a las parejas homosexuales?,https://www.ondacero.es/podcast/programas/julia-en-la-onda/gabinete/gabinete-que-implica-que-iglesia-acepte-bendecir...,es,2023,12
4,31f5c9e0f3db1ddf5d5938ee0c28ceb644f5be50e8bbada9e0473fc8a27d2bd9,2023-02-16,ondacero.es,ondacero.es,El Gabinete: El Congreso aprueba definitivamente la Ley Trans,https://www.ondacero.es/podcast/programas/julia-en-la-onda/gabinete/gabinete-congreso-aprueba-definitivamente-ley-tr...,es,2023,2


## 6. Diagnóstico rápido del dataset

In [7]:
print('Noticias totales:', len(df))
print('URLs únicas:', df['url'].nunique())
print('Medios únicos:', df['media_name'].nunique())
print('Fechas mín / máx:', df['publish_date'].min(), '→', df['publish_date'].max())

df['anio'].value_counts().sort_index()

Noticias totales: 79141
URLs únicas: 79141
Medios únicos: 233
Fechas mín / máx: 2023-01-01 00:00:00 → 2023-12-31 00:00:00


anio
2023    79141
Name: count, dtype: int64

## 7. Eliminar duplicados

MediaCloud puede devolver la misma URL más de una vez. Para extraer texto, no nos interesa repetir URLs.

In [8]:
df = df.drop_duplicates(subset='url').copy()
df = df.dropna(subset=['url', 'publish_date', 'title'])

print('Filas después de eliminar duplicados:', len(df))

Filas después de eliminar duplicados: 79141


## 8. Comprobar principales medios

In [9]:
df['media_name'].value_counts().head(20)

media_name
elperiodico.com                4199
diariosur.es                   3090
abc.es                         2520
elpais.com                     2412
publico.es                     1839
lavanguardia.com               1708
levante-emv.com                1702
eldiario.es                    1701
lne.es                         1565
diariodemallorca.es            1471
eldia.es                       1427
farodevigo.es                  1333
diariodeibiza.es               1260
elperiodico.cat                1257
elperiodicomediterraneo.com    1240
20minutos.es                   1239
lavozdigital.es                1220
europapress.es                 1148
elconfidencialdigital.com      1143
elespanol.com                  1101
Name: count, dtype: int64

## 9. Validación temática de la búsqueda

In [10]:
# 9. Muestra aleatoria de titulares

df['title'].sample(20, random_state=42).tolist()

['Diez curiosidades del Ford Ranger Raptor',
 'El menguante margen de libertad de la disidencia en Marruecos',
 'Abascal dice que su partido «protege mejor» que Sánchez y Macron a los homosexuales',
 'Los mejores churros del mundo...',
 'Deja de molestar a la gente de bien, pollo',
 'Pilar García Muñiz: "La ley Trans no aclara cómo es su aplicación en unas oposiciones con pruebas físicas"',
 'El gobierno dividido vuelve a Washington',
 'Madrid sin Fronteras 06.05.2023',
 'Feijóo abre las puertas a cargos y votantes de Cs para reforzar el centroderecha',
 'El directivo de tabloides amigo de Trump declara de nuevo ante el gran jurado',
 'Donald Tusk y los desafíos de un relevo en el poder de Polonia aún virtual',
 'Guerra vuelve a atacar a Yolanda Díaz y la acusa de estar siempre "en la peluquería"',
 '«Las autopistas necesitan una modernización para ser más sostenibles e impulsar la nueva movilidad»',
 "Collboni obre la porta a un pacte amb comuns i ERC per a arrabassar-li l'alcaldia a 

## 10. Filtrado temático preliminar

terminos = [
    'inmigr',
    'migrante',
    'migración',
    'refugiado',
    'asilo',
    'patera'
]

In [11]:
terminos_lgtb = [
    'lgtb',
    'lgtbi',
    'lgbt',
    r'\bgay\b',
    'gais',
    'lesbiana',
    'lesbianas',
    'homosexual',
    'homosexuales',
    'bisexual',
    'bisexuales',
    'transexual',
    'transexuales',
    'transgénero',
    'homofobia',
    'transfobia',
    'lgtbifobia',
    'orgullo gay',
    'orgullo lgtbi'
]

patron = '|'.join(terminos_lgtb)

df_filtrado = df[
    df['title'].str.lower().str.contains(
        patron,
        na=False
    )
].copy()

print("Original:", len(df))
print("Filtrado:", len(df_filtrado))

Original: 79141
Filtrado: 8635


In [12]:
df_filtrado["title"].sample(
    20,
    random_state=42
).tolist()

['Un ex actor porno gay, candidato del PP en un pueblo de Albacete',
 'Uganda aprueba una ley anti-homosexualidad',
 'Sánchez se enfrenta al salto generacional: «Pedro, tienes el gusto de un homosexual deprimido»',
 'La família d\'Itziar Castro reacciona al condol de Felip i Letícia, "republicana y lesbiana..."',
 'El checo Jantko, primer futbolista de la Liga que revela su homosexualidad',
 'Javier Ambrossi se sincera con Joaquín: “Me di cuenta de que era gay viendo Farmacia de Guardia”',
 'Mario Vaquerizo indigna el col·lectiu LGTBI a pocs dies de la seva participació a l’Atlantic Pride de la Corunya: «És repugnant»',
 'Canarias Orgullosa pone el foco en los creadores LGTBI',
 'Amnistía exige a Túnez la liberación de dos personas LGBTQ encarceladas por la ley contra la homosexualidad',
 '¿Bandera o pancarta? La consideración de la enseña LGTBI, pendiente del Supremo',
 'Francia estudia «reparar un error» al indemnizar a los homosexuales condenados entre 1942 y 1982',
 'Hay una lesbia

In [13]:
print(df_filtrado.shape)

(8635, 9)


## 11. Función para extraer texto con Trafilatura

In [14]:
def extraer_texto_trafilatura(url, pausa=0.5):

    try:
        descargado = trafilatura.fetch_url(url)
        time.sleep(pausa)

        if descargado is None:
            return None

        texto = trafilatura.extract(
            descargado,
            include_comments=False,
            include_tables=False,
            favor_precision=True
        )

        if texto is None:
            return None

        texto = texto.strip()

        if len(texto) < 300:
            return None

        return texto

    except Exception as e:
        return None

BATCH_SIZE = 500
START = 12500

for inicio in range(START, len(df_filtrado), BATCH_SIZE):

    fin = min(inicio + BATCH_SIZE, len(df_filtrado))

    bloque = df_filtrado.iloc[inicio:fin].copy()

    bloque["texto"] = bloque["url"].progress_apply(
        extraer_texto_trafilatura
    )

    bloque.to_csv(
        RUTA_SALIDA,
        mode="a",
        header=False,
        index=False
    )

    exito_bloque = bloque["texto"].notna().mean() * 100

    print(f"Guardado bloque {inicio}-{fin}")
    print(f"Éxito bloque: {exito_bloque:.1f}%")

In [15]:
BATCH_SIZE = 500

for inicio in range(0, len(df_filtrado), BATCH_SIZE):

    fin = min(inicio + BATCH_SIZE, len(df_filtrado))

    print(f"\nProcesando noticias {inicio:,} - {fin:,}")

    bloque = df_filtrado.iloc[inicio:fin].copy()

    bloque["texto"] = bloque["url"].progress_apply(
        extraer_texto_trafilatura
    )

    modo = "w" if inicio == 0 else "a"

    bloque.to_csv(
        RUTA_SALIDA,
        mode=modo,
        header=(inicio == 0),
        index=False
    )

    exito_bloque = bloque["texto"].notna().mean() * 100

    print(f"Éxito bloque: {exito_bloque:.1f}%")
    print(f"Guardado hasta fila {fin:,}")


Procesando noticias 0 - 500


100%|██████████| 500/500 [12:33<00:00,  1.51s/it]  


Éxito bloque: 85.4%
Guardado hasta fila 500

Procesando noticias 500 - 1,000


100%|██████████| 500/500 [11:44<00:00,  1.41s/it]


Éxito bloque: 87.8%
Guardado hasta fila 1,000

Procesando noticias 1,000 - 1,500


...

Éxito bloque: 86.6%
Guardado hasta fila 8,000

Procesando noticias 8,000 - 8,500


100%|██████████| 500/500 [08:45<00:00,  1.05s/it]


Éxito bloque: 89.0%
Guardado hasta fila 8,500

Procesando noticias 8,500 - 8,635


100%|██████████| 135/135 [02:37<00:00,  1.17s/it]

Éxito bloque: 90.4%
Guardado hasta fila 8,635


## 12. Decisión metodológica

Antes de continuar, interpreta la tasa de extracción:

- **Más de 70%** → podemos seguir con Trafilatura.
- **Entre 40% y 70%** → se puede seguir, pero revisando medios problemáticos.
- **Menos de 40%** → conviene buscar otra estrategia antes de extraer todo.


In [16]:
df_resultado = pd.read_csv(RUTA_SALIDA)

print(df_resultado.shape)

(8635, 10)


In [17]:
exito_final = (
    df_resultado["texto"]
    .notna()
    .mean()
    * 100
)

print(f"Éxito final: {exito_final:.2f}%")

Éxito final: 87.49%


In [18]:
df_resultado["longitud_texto"] = (
    df_resultado["texto"]
    .fillna("")
    .str.len()
)

df_resultado["longitud_texto"].describe()

count     8635.000000
mean      3036.862884
std       2822.684093
min          0.000000
25%       1214.000000
50%       2599.000000
75%       4299.500000
max      90989.000000
Name: longitud_texto, dtype: float64

In [19]:
df_resultado["url"].duplicated().sum()

np.int64(0)

## 13. Extracción completa

Ejecutar esta celda solo si la prueba piloto ha funcionado bien.

Puede tardar bastante con miles de URLs.

In [ ]:
df_resultado = pd.read_csv(RUTA_SALIDA)

print(df_resultado.shape)

exito = df_resultado["texto"].notna().mean() * 100
print(f"Éxito final: {exito:.2f}%")

(8635, 10)
Éxito final: 87.49%


In [21]:
fallos = df_lgtb_2015[df_lgtb_2015["texto"].isna()].copy()

print("Fallos:", len(fallos))

fallos[["media_name", "title"]].head(30)

Fallos: 1080


,media_name,title
1,melillahoy.es,El PSOE ve “muy mal” la ausencia de CPM en el Día del Orgullo LGTBI
6,noticiasdenavarra.com,Polémica por los brazaletes LGTBI de 'Black Eyed Peas' en el especial de Nochevieja de la televisión polaca
8,rioja2.com,"Fallece Ángel Blasco, histórico activista riojano por los derechos LGTBI+"
12,noticiasdenavarra.com,EEUU llevará a cabo este martes la primera ejecución de una mujer transexual en el país
19,abc.es,EE.UU. llevará a cabo este martes la primera ejecución de una transexual en el país
50,clm24.es,Detenido un policía acusado de drogar y violar brutalmente a una mujer transexual
53,elfaro.es,La homofobia da la bienvenida al 2023
54,elpais.com,"“Mamá, soy lesbiana”: cómo hablar con los adolescentes sobre su orientación sexual"
56,elpais.com,"Noah Schnapp, de ‘Stranger Things’: “Finalmente, les dije a mis amigos y familiares que era gay”"
66,naciodigital.cat,Noah Schnapp i el perquè encara hi ha persones «atemorides» de dir que són gais en ple segle XXI


In [22]:
fallos["media_name"].value_counts().head(20)

media_name
elpais.com                172
abc.es                    123
diariosur.es               88
noticiasdenavarra.com      80
noticierouniversal.com     58
eldiadigital.es            43
diariocritico.com          39
clm24.es                   34
publico.es                 30
naciodigital.cat           27
cope.es                    21
salamancartvaldia.es       17
diario16.com               16
rioja2.com                 16
levante-emv.com            13
madridiario.es             11
estrelladigital.es         11
elcorreo.com                9
lavozdigital.es             9
elimparcial.es              9
Name: count, dtype: int64

In [23]:
# Ejecutar solo cuando decidamos continuar

# df['texto'] = df['url'].progress_apply(extraer_texto_trafilatura)

# df['texto_extraido'] = df['texto'].notna()
# print('Tasa final de extracción:', df['texto_extraido'].mean() * 100)

# df.head()

## 14. Guardar dataset procesado

In [24]:
# Ejecutar después de la extracción completa

# df.to_csv(RUTA_SALIDA, index=False)
# print('Archivo guardado en:', RUTA_SALIDA)

## 15. Comprobación final

In [25]:
# df_final = pd.read_csv(RUTA_SALIDA)
# print(df_final.shape)
# df_final[['publish_date', 'media_name', 'title', 'url', 'texto']].head()